In [1]:
import glob
import pandas as pd

csv_files = glob.glob('*.csv')

for filename in csv_files:
    print(f"📄 文件名: {filename}")
    try:
        # 只读取前 2 行 (nrows=1 读取数据行，header默认是第一行)
        # 注意：pd.read_csv 默认把第一行当做表头，nrows=1 会读入表头 + 1行数据
        df = pd.read_csv(filename, nrows=1)
        
        # 打印表头（列名）
        print(f"   第一行 (列名): {list(df.columns)}")

        # 打印第一行数据 (如果文件不为空)
        if not df.empty:
            print(f"   第二行 (数值): {df.iloc[0].tolist()}")

    except Exception as e:
        print(f"   读取出错: {e}")
    print("-" * 40)

📄 文件名: control.csv
   第一行 (列名): ['Image Data ID', 'Unnamed: 1', 'Subject', 'Group', 'Sex', 'Age', 'Visit', 'Description', 'Type', 'Acq Date', 'Format', 'visit_month', 'code_upd2301_speech_problems', 'code_upd2302_facial_expression', 'code_upd2303a_rigidity_neck', 'code_upd2303b_rigidity_rt_upper_extremity', 'code_upd2303c_rigidity_left_upper_extremity', 'code_upd2303d_rigidity_rt_lower_extremity', 'code_upd2303e_rigidity_left_lower_extremity', 'code_upd2304a_right_finger_tapping', 'code_upd2304b_left_finger_tapping', 'code_upd2305a_right_hand_movements', 'code_upd2305b_left_hand_movements', 'code_upd2306a_pron_sup_movement_right_hand', 'code_upd2306b_pron_sup_movement_left_hand', 'code_upd2307a_right_toe_tapping', 'code_upd2307b_left_toe_tapping', 'code_upd2308a_right_leg_agility', 'code_upd2308b_left_leg_agility', 'code_upd2309_arising_from_chair', 'code_upd2310_gait', 'code_upd2311_freezing_of_gait', 'code_upd2312_postural_stability', 'code_upd2313_posture', 'code_upd2314_body_bradyk

In [2]:
import pandas as pd
import numpy as np

# ================= ⚡ 配置区 =================
RAW_KG_PATH = "kg.csv"
OUTPUT_PATH = "primekg_parkinson_metabolic.csv"

# 黑名单 (保持不变)
BLACKLIST_NODES = [
    "multi-cellular organism", "small intestine", "intestine", "colon", "stomach",
    "testis", "fallopian tube", "prostate gland", "spleen", "adrenal gland",
    "adult mammalian kidney", "cortex of kidney", "dorsolateral prefrontal cortex"
]

# ★★★ 策略升级：将关键词分为“核心”和“外围” ★★★

# 1. 核心关键词 (允许扩展 PPI)
# 只有跟这些词沾边的基因，才有资格去拉帮结派
CORE_KEYWORDS = [
    "Parkinson", "Lewy body", "Synuclein", "Substantia nigra", "Dopamine", "Basal ganglia", # 病理
    "Tremor", "Rigidity", "Bradykinesia", "Gait", "Posture", "Instability", "Freezing"      # 症状
]

# 2. 外围关键词 (只保留节点，禁止扩展 PPI)
# 这些词是为了挂载 CSV 数据的，不需要它们的整个基因家族
PERIPHERAL_KEYWORDS = [
    "Hypotension", "Diabetes", "Insulin", "Glucose", "Lipid", "Cholesterol"
]

# 合并用于初步筛选
ALL_KEYWORDS = CORE_KEYWORDS + PERIPHERAL_KEYWORDS

VALID_TYPES = ['disease', 'gene/protein', 'effect/phenotype', 'pathway', 'biological_process', 'anatomy']

def main():
    print(f"🔹 1. 加载 PrimeKG ({RAW_KG_PATH}) ...")
    df = pd.read_csv(RAW_KG_PATH, low_memory=False)
    
    # --- 基础清洗 ---
    print(f"🔹 2. 基础清洗...")
    mask_type = (df['x_type'].isin(VALID_TYPES)) & (df['y_type'].isin(VALID_TYPES))
    mask_rel = ~df['relation'].isin(['indication', 'contraindication', 'drug_drug', 'off-label use'])
    mask_black = (~df['x_name'].isin(BLACKLIST_NODES)) & (~df['y_name'].isin(BLACKLIST_NODES))
    df_clean = df[mask_type & mask_rel & mask_black].copy()

    # --- 关键词提取 (核心 + 外围) ---
    print(f"🔹 3. 提取核心与外围知识...")
    pattern = '|'.join(ALL_KEYWORDS)
    mask_key = (df_clean['x_name'].str.contains(pattern, case=False, na=False)) | \
               (df_clean['y_name'].str.contains(pattern, case=False, na=False))
    
    subset = df_clean[mask_key].copy()
    print(f"   ✅ 基础骨架构建完成: {len(subset)} 条")
    
    # ================= ⚡ 智能 PPI 扩展逻辑 ⚡ =================
    print("\n🔹 4. 执行智能 PPI 扩展 (仅针对核心神经基因)...")
    
    # 步骤 A: 找出所有“核心相关”的基因
    # 逻辑：如果一个基因连到了 "Parkinson" 或 "Tremor"，它就是 VIP
    core_pattern = '|'.join(CORE_KEYWORDS)
    mask_core_rows = (subset['x_name'].str.contains(core_pattern, case=False)) | \
                     (subset['y_name'].str.contains(core_pattern, case=False))
    
    core_subset = subset[mask_core_rows]
    
    # 提取 VIP 基因名单
    vip_genes = set(core_subset[core_subset['y_type'] == 'gene/protein']['y_name']) | \
                set(core_subset[core_subset['x_type'] == 'gene/protein']['x_name'])
    
    print(f"   🧬 识别到 {len(vip_genes)} 个核心神经/PD基因 (VIP)")
    print(f"      (只有这 {len(vip_genes)} 个基因的互作会被保留，剔除了通用代谢基因的干扰)")

    if len(vip_genes) > 0:
        # 步骤 B: 只查找 VIP 基因之间的互作
        mask_ppi = (
            (df_clean['x_name'].isin(vip_genes)) &
            (df_clean['y_name'].isin(vip_genes)) &
            (df_clean['relation'] == 'protein_protein')
        )
        ppi_df = df_clean[mask_ppi]
        print(f"   🕸 补充精选 PPI 边: {len(ppi_df)} 条 (原方案为 39万条)")
        
        final_df = pd.concat([subset, ppi_df]).drop_duplicates()
    else:
        final_df = subset

    # --- 保存 ---
    print(f"\n🔹 5. 保存结果至: {OUTPUT_PATH}")
    final_df.to_csv(OUTPUT_PATH, index=False)
    
    print("-" * 50)
    print(f"🎉 优化完成！总条目: {len(final_df)}")
    print("   优化策略: 保留了代谢节点用于连接 CSV，但切断了它们的冗余基因网络。")
    print("-" * 50)

if __name__ == "__main__":
    main()

🔹 1. 加载 PrimeKG (kg.csv) ...
🔹 2. 基础清洗...
🔹 3. 提取核心与外围知识...
   ✅ 基础骨架构建完成: 50758 条

🔹 4. 执行智能 PPI 扩展 (仅针对核心神经基因)...
   🧬 识别到 11972 个核心神经/PD基因 (VIP)
      (只有这 11972 个基因的互作会被保留，剔除了通用代谢基因的干扰)
   🕸 补充精选 PPI 边: 370074 条 (原方案为 39万条)

🔹 5. 保存结果至: primekg_parkinson_metabolic.csv
--------------------------------------------------
🎉 优化完成！总条目: 420832
   优化策略: 保留了代谢节点用于连接 CSV，但切断了它们的冗余基因网络。
--------------------------------------------------


In [3]:
import pandas as pd

# ================= 配置区 =================
KG_FILE = "primekg_parkinson_metabolic.csv"

def inspect_quality():
    print(f"🚀 正在加载 {KG_FILE} 进行质检...")
    try:
        df = pd.read_csv(KG_FILE)
    except FileNotFoundError:
        print("❌ 找不到文件！")
        return

    print(f"📊 数据总量: {len(df)} 条")
    print("-" * 50)

    # 1. 检查节点“贫富差距” (寻找超级 Hub)
    # 如果排在前面的全是 "Ubiquitin B" 这种通用蛋白，说明噪音很大
    all_nodes = pd.concat([df['x_name'], df['y_name']])
    top_nodes = all_nodes.value_counts().head(15)
    
    print("🏆 [Top 15 最活跃节点] (如果出现通用蛋白，考虑从关键词移除):")
    for node, count in top_nodes.items():
        print(f"   - {node:<30} : {count:>5} 连线")
    print("-" * 50)

    # 2. 检查关键症状的覆盖情况 (CSV 挂载点)
    # 必须确保 Tremor, Rigidity, Diabetes 等在里面
    check_list = [
        "Tremor", "Rigidity", "Bradykinesia", "Gait", # 运动症状
        "Hypotension", "Orthostatic hypotension",     # 血压
        "Diabetes", "Insulin", "Lipid"                # 代谢
    ]
    
    print("🎯 [关键挂载点检查] (CSV数据能否连上?):")
    for kw in check_list:
        # 模糊搜索
        mask = (df['x_name'].str.contains(kw, case=False)) | \
               (df['y_name'].str.contains(kw, case=False))
        count = len(df[mask])
        status = "✅ 丰富" if count > 50 else ("⚠️ 稀少" if count > 0 else "❌ 缺失")
        print(f"   - {kw:<25} : {count:>4} 条 -> {status}")
        
        # 如果有数据，打印一条示例看看连到了谁
        if count > 0:
            sample = df[mask].iloc[0]
            print(f"     (示例: {sample['x_name']} --[{sample['relation']}]--> {sample['y_name']})")
            
    print("-" * 50)

    # 3. 检查 PPI (基因互作) 的质量
    # 看看被判定为 VIP 的基因到底是什么
    print("🧬 [PPI 互作质量抽查]:")
    ppi_df = df[df['relation'] == 'protein_protein']
    print(f"   PPI 总数: {len(ppi_df)} 条 (占总数的 {len(ppi_df)/len(df):.1%})")
    
    if len(ppi_df) > 0:
        print("   随机抽查 5 条 PPI:")
        print(ppi_df[['x_name', 'y_name']].sample(5).to_string(index=False))

if __name__ == "__main__":
    inspect_quality()

🚀 正在加载 primekg_parkinson_metabolic.csv 进行质检...


C:\Users\86137\AppData\Local\Temp\ipykernel_11048\98977645.py:9: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(KG_FILE)


📊 数据总量: 420832 条
--------------------------------------------------
🏆 [Top 15 最活跃节点] (如果出现通用蛋白，考虑从关键词移除):
   - substantia nigra               : 23730 连线
   - UBC                            :  8548 连线
   - ETS1                           :  2434 连线
   - GATA2                          :  2052 连线
   - AR                             :  1862 连线
   - CTCF                           :  1802 连线
   - MYC                            :  1686 连线
   - EP300                          :  1536 连线
   - APP                            :  1516 连线
   - SUMO2                          :  1428 连线
   - EGR1                           :  1356 连线
   - RAD21                          :  1292 连线
   - TP53                           :  1236 连线
   - CTNNB1                         :  1172 连线
   - DISC1                          :  1156 连线
--------------------------------------------------
🎯 [关键挂载点检查] (CSV数据能否连上?):
   - Tremor                    : 1634 条 -> ✅ 丰富
     (示例: ACHE --[phenotype_protein]--> Tremor)
   - Rigidity   

In [4]:
import pandas as pd

# 配置
INPUT_FILE = "primekg_parkinson_metabolic.csv"
OUTPUT_FILE = "primekg_parkinson_final.csv"

# 🛑 噪音黑名单 (基于你的质检结果)
NOISE_NODES = [
    "UBC", "SUMO2",       # 通用泛素化修饰
    "GATA2", "ETS1", "AR", "CTCF", "MYC", "EP300", "TP53", # 通用转录因子
    "APP", "CTNNB1"       # 虽然相关，但如果是Hub且连接过泛，建议限制(可选，这里先删最大的UBC)
]
# 注意：我们只删最大的几个 Hub，保留 APP 因为它和神经退行性疾病关系紧密

def clean():
    print(f"🧹 正在清洗 {INPUT_FILE} ...")
    df = pd.read_csv(INPUT_FILE)
    original_len = len(df)
    
    # 1. 剔除噪音节点 (不管是做 head 还是 tail)
    mask_noise = (df['x_name'].isin(NOISE_NODES)) | (df['y_name'].isin(NOISE_NODES))
    df_clean = df[~mask_noise].copy()
    
    # 2. 再次去重
    df_clean.drop_duplicates(inplace=True)
    
    new_len = len(df_clean)
    print(f"   ✂️ 移除了 {original_len - new_len} 条涉及通用Hub的连线")
    print(f"   📊 最终图谱规模: {new_len} 条 (黄金区间: 5w~15w)")
    
    df_clean.to_csv(OUTPUT_FILE, index=False)
    print(f"✅ 已保存至: {OUTPUT_FILE}")

if __name__ == "__main__":
    clean()

🧹 正在清洗 primekg_parkinson_metabolic.csv ...


C:\Users\86137\AppData\Local\Temp\ipykernel_11048\2710800288.py:17: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(INPUT_FILE)


   ✂️ 移除了 25204 条涉及通用Hub的连线
   📊 最终图谱规模: 395628 条 (黄金区间: 5w~15w)
✅ 已保存至: primekg_parkinson_final.csv
